# HyperDream Colab 自动回传结果到 GitHub（分步大白话版）

这份 Notebook 解决的问题：
- 你在 Colab 跑完实验后，结果自动 push 到 GitHub。
- 我这边就能直接读取你最新结果并分析，不用你手动复制粘贴大量日志。

```text
整体流程（Flow）
├─ 1) 挂载 Drive（可选）
├─ 2) 拉代码
├─ 3) 装依赖
├─ 4) 设置实验参数
├─ 5) 安全输入 GitHub Token
├─ 6) 跑实验
├─ 7) 自动 push 结果到 GitHub 分支
└─ 8) 打印分支和提交号
```

注意：Token 只放环境变量，不要打印出来。

## Cell 1：挂载 Google Drive（可选）

大白话：
- 防止 Colab 重置后结果丢失。
- 有 Drive 更稳。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## Cell 2：拉取项目（首次 clone，后续 pull）

大白话：
- 确保你跑的是最新版本代码。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/peter941221/High_Dimensional_WorldModel.git'
PROJECT_DIR = Path('/content/High_Dimensional_WorldModel')
BRANCH = 'main'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('当前目录:', Path.cwd())

## Cell 3：安装依赖

大白话：
- 没这一步，后面脚本可能报包缺失。

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## Cell 4：实验参数（你最常改这格）

大白话：
- `RUN_ID`：本次实验名。
- `RESUME`：是否续训。
- `PUSH_BRANCH`：结果推到哪个分支（建议 `colab-results`）。

In [ ]:
from datetime import datetime

RUN_ID = f'colab_push_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RESUME = False

BASELINE_EPOCHS = 3
TRANSFER_PRETRAIN_EPOCHS = 2
TRANSFER_FINETUNE_EPOCHS = 2
ABLATION_EPOCHS = 2
ROBUSTNESS_EPISODES = 20
EVAL_EPISODES = 10
MAX_STEPS = 80

SAVE_EVERY = 2
KEEP_LAST = 3

RUN_TESTS = True
SYNC_TO_DRIVE = True
DRIVE_SYNC_DIR = '/content/drive/MyDrive/High_Dimensional_WorldModel_runs'

PUSH_RESULTS_TO_GITHUB = True
PUSH_BRANCH = 'colab-results'
GITHUB_USER = 'peter941221'
REPO_NAME = 'High_Dimensional_WorldModel'
TOKEN_ENV = 'GITHUB_TOKEN'

print('RUN_ID =', RUN_ID)
print('RESUME =', RESUME)
print('PUSH_BRANCH =', PUSH_BRANCH)

## Cell 5：安全输入 GitHub Token（必要）

大白话：
- 这里输入你的 Personal Access Token（PAT）。
- 只存到环境变量，不会打印出来。

Token 权限建议（最小化）：
- private repo：`repo`
- public repo：`public_repo`

如果你不想自动 push，可把 `PUSH_RESULTS_TO_GITHUB=False`。

In [ ]:
import getpass
import os

if PUSH_RESULTS_TO_GITHUB:
    token = getpass.getpass('请输入 GitHub PAT（输入不可见）: ').strip()
    os.environ[TOKEN_ENV] = token
    print(f'✅ Token 已写入环境变量: {TOKEN_ENV}')
else:
    print('跳过 Token 输入（PUSH_RESULTS_TO_GITHUB=False）')

## Cell 6：先跑测试（推荐）

大白话：
- 先确保代码健康，再花时间跑训练。

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
else:
    print('跳过测试')

## Cell 7：一键跑实验 + 可选同步到 Drive + 自动 push 到 GitHub

大白话：
- 这里调用 `colab_autorun.py`。
- 它会执行：实验训练 -> 出图 ->（可选）Drive同步 ->（可选）GitHub push。

In [ ]:
import subprocess

cmd = [
    'python', 'colab_autorun.py',
    '--run-id', RUN_ID,
    '--baseline-epochs', str(BASELINE_EPOCHS),
    '--transfer-pretrain-epochs', str(TRANSFER_PRETRAIN_EPOCHS),
    '--transfer-finetune-epochs', str(TRANSFER_FINETUNE_EPOCHS),
    '--ablation-epochs', str(ABLATION_EPOCHS),
    '--robustness-episodes', str(ROBUSTNESS_EPISODES),
    '--eval-episodes', str(EVAL_EPISODES),
    '--max-steps', str(MAX_STEPS),
    '--save-every', str(SAVE_EVERY),
    '--keep-last', str(KEEP_LAST),
]

if RESUME:
    cmd.append('--resume')
if RUN_TESTS:
    cmd.append('--run-tests')
if SYNC_TO_DRIVE:
    cmd.extend(['--sync-to-drive', '--mount-drive', '--drive-sync-dir', DRIVE_SYNC_DIR])

if PUSH_RESULTS_TO_GITHUB:
    cmd.extend([
        '--push-results-to-github',
        '--push-branch', PUSH_BRANCH,
        '--github-user', GITHUB_USER,
        '--repo-name', REPO_NAME,
        '--token-env', TOKEN_ENV,
    ])

print('将执行命令:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('✅ 自动流水线执行完成')

## Cell 8：检查 Git 分支与最新提交

大白话：
- 你要确认结果有没有真的 push 上去。
- 看 `colab-results` 分支是否有新提交。

In [ ]:
!git branch -a
!git log --oneline -n 10
!echo '结果分支（网页）: https://github.com/peter941221/High_Dimensional_WorldModel/tree/colab-results'

## Cell 9：下次续训怎么做？

大白话：
1. `RUN_ID` 用上次同一个。
2. `RESUME=True`。
3. 把 epoch 调大（例如 3 -> 10）。

这样会接着跑，不会重头开始。

## Cell 10：如果 push 失败怎么办？

常见原因：
- PAT 权限不够
- Token 过期
- 仓库用户名/仓库名写错
- 分支受保护（branch protection）

排查顺序（建议）：
1. 先确认 Token 权限。
2. 再确认 `GITHUB_USER/REPO_NAME/PUSH_BRANCH`。
3. 重新运行 Cell 5 + Cell 7。